# combine

> Combine calibrated dithers into a single mosaic using swarp.

In [ ]:
# | default_exp euclid.combine

In [ ]:
# | exporti

import os
import tempfile
from pathlib import Path

import numpy as np
from astropy.io import fits
from astropy.stats import SigmaClip
from photutils.background import Background2D

from nicl.mask import fast_mask

In [ ]:
# | exporti

swarp_config_nisp = """
#----------------------------------- Output -----------------------------------
IMAGEOUT_NAME          coadd.fits      # Output filename
WEIGHTOUT_NAME         coadd.weight.fits # Output weight-map filename
 
HEADER_ONLY            N               # Only a header as an output file (Y/N)?
HEADER_SUFFIX          .head           # Filename extension for additional headers
 
#------------------------------- Input Weights --------------------------------

WEIGHT_TYPE            MAP_WEIGHT    # BACKGROUND,MAP_RMS,MAP_VARIANCE
                                       # or MAP_WEIGHT
RESCALE_WEIGHTS        N               # Rescale input weights/variances (Y/N)?
WEIGHT_SUFFIX          .weight.fits    # Suffix to use for weight-maps
WEIGHT_IMAGE                           # Weightmap filename if suffix not used
                                       # (all or for each weight-map)
WEIGHT_THRESH                          # Bad pixel weight-threshold
 
#------------------------------- Co-addition ----------------------------------
 
COMBINE                Y               # Combine resampled images (Y/N)?
COMBINE_TYPE           WEIGHTED         # MEDIAN,AVERAGE,MIN,MAX,WEIGHTED,CLIPPED
                                       # CHI-OLD,CHI-MODE,CHI-MEAN,SUM,
                                       # WEIGHTED_WEIGHT,MEDIAN_WEIGHT,
                                       # AND,NAND,OR or NOR
#CLIP_AMPFRAC           0.3             # Fraction of flux variation allowed
                                       # with clipping
#CLIP_SIGMA             4.0             # RMS error multiple variation allowed
                                       # with clipping
#CLIP_WRITELOG          N               # Write output file with coordinates of
                                       # clipped pixels (Y/N)
#CLIP_LOGNAME           clipped.log     # Name of output file with coordinates
                                       # of clipped pixels
#BLANK_BADPIXELS        N               # Set to 0 pixels having a weight of 0
 
#-------------------------------- Astrometry ----------------------------------
 
CELESTIAL_TYPE         NATIVE          # NATIVE, PIXEL, EQUATORIAL,
                                       # GALACTIC,ECLIPTIC, or SUPERGALACTIC
PROJECTION_TYPE        TAN             # Any WCS projection code or NONE
PROJECTION_ERR         0.001           # Maximum projection error (in output
                                       # pixels), or 0 for no approximation
CENTER_TYPE            ALL             # MANUAL, ALL or MOST
CENTER         00:00:00.0, +00:00:00.0 # Coordinates of the image center
PIXELSCALE_TYPE        MANUAL          # MANUAL,FIT,MIN,MAX or MEDIAN
PIXEL_SCALE            0.3             # Pixel scale
IMAGE_SIZE             0               # Image size (0 = AUTOMATIC)
 
#-------------------------------- Resampling ----------------------------------
 
RESAMPLE               Y               # Resample input images (Y/N)?
RESAMPLE_DIR           .               # Directory path for resampled images
RESAMPLE_SUFFIX        .resamp.fits    # filename extension for resampled images
 
RESAMPLING_TYPE        LANCZOS2        # NEAREST,BILINEAR,LANCZOS2,LANCZOS3
                                       # LANCZOS4 (1 per axis) or FLAGS
OVERSAMPLING           0               # Oversampling in each dimension
                                       # (0 = automatic)
INTERPOLATE            N               # Interpolate bad input pixels (Y/N)?
                                       # (all or for each image)
 
FSCALASTRO_TYPE        VARIABLE            # NONE,FIXED, or VARIABLE
FSCALE_KEYWORD         PHOSCALE            # FITS keyword for the multiplicative
                                       # factor applied to each input image
FSCALE_DEFAULT         1.0             # Default FSCALE value if not in header
 
GAIN_KEYWORD           GAIN            # FITS keyword for effect. gain (e-/ADU)
GAIN_DEFAULT           0.0             # Default gain if no FITS keyword found

SATLEV_KEYWORD         SATURATE        # FITS keyword for saturation level (ADU)
SATLEV_DEFAULT         50000.0         # Default saturation if no FITS keyword

#--------------------------- Background subtraction ---------------------------
 
SUBTRACT_BACK          N               # Subtraction sky background (Y/N)?
                                       # (all or for each image)
 
BACK_TYPE              AUTO            # AUTO or MANUAL
                                       # (all or for each ima  ge)
BACK_DEFAULT           0.0             # Default background value in MANUAL
                                       # (all or for each image)
BACK_SIZE              32             # Background mesh size (pixels)
                                       # (all or for each image)
BACK_FILTERSIZE        5               # Background map filter range (meshes)
                                       # (all or for each image)
 
#------------------------------ Memory management -----------------------------
 
VMEM_DIR               .               # Directory path for swap files
VMEM_MAX               6144            # Maximum amount of virtual memory (MB)
MEM_MAX                6144            # Maximum amount of usable RAM (MB)
COMBINE_BUFSIZE        6144            # RAM dedicated to co-addition(MB)
 
#------------------------------ Miscellaneous ---------------------------------
 
DELETE_TMPFILES        Y               # Delete temporary resampled FITS files
                                       # (Y/N)?
COPY_KEYWORDS          RA, DEC, DATAUNIT, DET_ID, ZPAB, ZPABE, ZPVEGA, ZPVEGAE, OBJECT # List of FITS keywords to propagate
                                       # from the input to the output headers
WRITE_FILEINFO         N               # Write information about each input
                                       # file in the output image header?
WRITE_XML              Y               # Write XML file (Y/N)?
XML_NAME               swarp.xml       # Filename for XML output
VERBOSE_TYPE           NORMAL          # QUIET,LOG,NORMAL, or FULL
 
NTHREADS               0               # Number of simultaneous threads for
                                       # the SMP version of SWarp
                                       # 0 = automatic
"""

# The VIS config may need some customisations
swarp_config_vis = swarp_config_nisp

In [ ]:
# | export


class Combiner:
    """Combines Euclid calibrated dithers."""

    def __init__(
        self,
        in_path="./input_images",  # path at which to find the input images
        out_path="./output_images",  # path at which to place output images
        obs_ids=None,  # default obs_ids to process (int, string or iterable), if not specified further
        filters=[
            "VIS",
            "Y",
            "J",
            "H",
        ],  # default filters to process, if not specified further
        name=None,  # default name for the output image
        bkg_sub=True,  # subtract background
        bkg_mesh_size=None,  # size of the background mesh
        bits_to_mask=None,  # list of DQ bits to mask
        overwrite=False,  # overwrite existing files
        debug=False,  # retain intermediate files for checking
    ):
        """Create an object for accessing data and log in to the ESA server."""
        self.in_path = Path(in_path)
        self.out_path = Path(out_path)
        self.obs_ids = obs_ids
        self.filters = filters
        self.name = name
        self.overwrite = overwrite
        self.debug = debug
        if bits_to_mask is not None:
            self.bits_to_mask = bits_to_mask
        else:
            self.bits_to_mask = [2, 3, 4, 6, 8, 9, 10, 12, 13, 16, 18, 23]

    def combine(
        self,
        obs_ids=None,  # obs_ids to process (int, string or iterable)
        filters=None,  # filters to process ("VIS", "Y", "J", "H", or iterable)
        name=None,  # name for the output image
    ):
        if obs_ids is None:
            obs_ids = self.obs_ids
        obs_ids = np.atleast_1d(obs_ids)
        if filters is None:
            filters = self.filters
        filters = np.atleast_1d(filters)
        if name is None:
            name = self.name
        for filt in filters:
            self.combine_filter(obs_ids, filt, name)

    def combine_filter(self, obs_ids, filt, name=None):
        dithers = self.get_dithers(obs_ids, filt)
        if len(dithers) == 0:
            return
        if filt == "VIS":
            det_ids = [f"{col}-{row}" for row in range(1, 7) for col in range(1, 7)]
            detectors = [f"CCDID{i}" for i in det_ids]
        else:
            det_ids = [f"{col}{row}" for row in range(1, 5) for col in range(1, 5)]
            detectors = [f"DET{i}" for i in det_ids]
        out_fn = dithers[0].name.split("-")[0] + "-STK-IMAGE" + f"_{filt}"
        if name is None and len(obs_ids) >= 1:
            name = "-".join([str(obs_id) for obs_id in obs_ids])
        if name is not None:
            out_fn += f"-{name}"
        out_fn += ".fits"
        cwd = os.getcwd()
        try:
            with tempfile.TemporaryDirectory(delete=(not self.debug)) as tmpdirname:
                if self.debug:
                    print(f"Intermediate files can be found in {tmpdirname}/.")
                os.chdir(tmpdirname)
                tmp_fns_sci = self.prepare_sci_dithers(dithers, detectors)
                self.prepare_weight_dithers(dithers, detectors)
                with open("images.lis", "w") as file:
                    for fn in tmp_fns_sci:
                        file.write(f"{fn}[1]\n")
                with open("config.swarp", "w") as file:
                    if filt == "VIS":
                        file.write(swarp_config_vis)
                    else:
                        file.write(swarp_config_nisp)
                os.system("swarp @images.lis -c config.swarp")
                self.copy_swarp_to_output(out_fn)
        finally:
            os.chdir(cwd)

    def get_dithers(self, obs_ids, filt):
        dithers = []
        for obs_id in obs_ids:
            dithers.extend(
                self.in_path.glob(f"**/EUC_*CAL-IMAGE_{filt}-{obs_id}-*Z.fits")
            )
        n_dithers = len(dithers)
        n_det = 36 if filt == "VIS" else 4
        n_obs = len(obs_ids)
        if n_dithers == 0:
            print("Found no files.")
        elif n_dithers < n_det * n_obs:
            print("Missing some files.")
        elif n_dithers > n_det * n_obs:
            print("Found too many files.")
        return dithers

    def prepare_sci_dithers(self, dithers, detectors):
        print("Preparing science dithers for swarp...")
        tmp_fns = []
        for i, dither in enumerate(dithers):
            with fits.open(dither) as hdul:
                for det in detectors:
                    extension = f"{det}.SCI"
                    data = hdul[extension].data
                    primary_hdr = hdul[0].header
                    ext_hdr = hdul[extension].header
                    if self.bkg_sub:
                        dq = hdul[det + ".DQ"].data
                        obj_mask, _ = fast_mask(data, estimate_background=True)
                        bad_pix_mask = np.any(
                            [(dq & 2**bit > 0) for bit in self.bits_to_mask], axis=0
                        )
                        # combine the bad pixel mask and the object mask to form a final mask
                        mask = bad_pix_mask | obj_mask
                        # 10 x 10 mesh by default if not specified
                        if self.bkg_mesh_size is None:
                            self.bkg_mesh_size = (data.shape[0] // 10, data.shape[1] // 10)
                        bg = Background2D(data, self.bkg_mesh_size, mask=mask, filter_size=(3, 3), sigma_clip=SigmaClip(sigma=3), exclude_percentile=90.0)
                        # subtract the background and modify the header
                        data -= bg.background
                        primary_hdr.add_history(f"Background modeled and subtracted using {self.bkg_mesh_size} pixel boxes.")
                    if det.startswith("DET"):
                        exptime = primary_hdr["EXPTIME"]
                        photfnu = primary_hdr["PHOTFNU"]
                        phrelex = primary_hdr["PHRELEX"]
                        phreldt = ext_hdr["PHRELDT"]
                        phoscale = (1.0 / exptime) * photfnu * phrelex * phreldt
                        ext_hdr.append(
                            (
                                "PHOSCALE",
                                phoscale,
                                "Combined photometric scaling factors",
                            )
                        )
                    else:
                        # may need to subtract background from VIS images here
                        pass
                    frame = fits.HDUList([fits.PrimaryHDU(header=primary_hdr), fits.ImageHDU(data, ext_hdr)])
                    tmp_fn = f"sci_{i}_{det}.fits"
                    tmp_fns.append(tmp_fn)
                    frame.writeto(tmp_fn)
        return tmp_fns

    def prepare_weight_dithers(self, dithers, detectors):
        print("Preparing weight dithers for swarp...")
        for i, dither in enumerate(dithers):
            with fits.open(dither) as hdul:
                for det in detectors:
                    extension_rms = f"{det}.RMS"
                    extension_dq = (
                        f"{det}.DQ" if det.startswith("DET") else f"{det}.FLG"
                    )
                    frame = fits.HDUList([hdul[0], hdul[extension_rms]])
                    dq_data = hdul[extension_dq].data
                    with np.errstate(divide="ignore"):
                        np.divide(1.0, np.square(frame[1].data), out=frame[1].data)
                    combined_mask = np.any(
                        [(dq_data & 2**bit > 0) for bit in self.bits_to_mask], axis=0
                    )
                    frame[1].data[combined_mask] = 0
                    tmp_fn = f"sci_{i}_{det}.weight.fits"
                    frame.writeto(tmp_fn)

    def copy_swarp_to_output(self, out_fn):
        with fits.open("coadd.weight.fits", memmap=True) as hdul_weights:
            hdu_rms = fits.ImageHDU(hdul_weights[0].data, hdul_weights[0].header)
        with np.errstate(divide="ignore"):
            np.divide(1.0, hdu_rms.data, out=hdu_rms.data)
        hdu_rms.data[hdu_rms.data == np.inf] = 1.0e16
        np.sqrt(hdu_rms.data, out=hdu_rms.data)
        hdu_pri = fits.PrimaryHDU()
        hdul_rms = fits.HDUList([hdu_pri, hdu_rms])
        hdul_rms.writeto("rms.fits")
        with fits.open("coadd.fits", memmap=True) as hdul_sci:
            hdu_sci = fits.ImageHDU(hdul_sci[0].data, hdul_sci[0].header)
        hdu_sci.header.set("XTENSION", "IMAGE   ", "Image extension", before=True)
        hdu_sci.header.set("PCOUNT", 0, "number of parameters", after="NAXIS2")
        hdu_sci.header.set("GCOUNT", 1, "number of groups", after="PCOUNT")
        hdu_sci.header.set("EXTNAME", "SCI", "Extension name", after="GCOUNT")
        hdu_sci.header.set("ZPAB", 23.9)
        hdu_sci.header.set("ZPABE", 0.0)
        hdu_sci.header.set("ZPVEGA", 1.0)
        hdu_sci.header.set("ZPVEGAE", 0.0)
        hdu_rms.header.set("XTENSION", "IMAGE   ", "Image extension", before=True)
        hdu_rms.header.set("PCOUNT", 0, "number of parameters", after="NAXIS2")
        hdu_rms.header.set("GCOUNT", 1, "number of groups", after="PCOUNT")
        hdu_rms.header.set("EXTNAME", "RMS", "Extension name", after="GCOUNT")
        hdu_sci.header.remove("SIMPLE", ignore_missing=True)
        for key in ("SIMPLE", "ZPAB", "ZPAB", "ZPVEGA", "ZPVEGAE"):
            hdu_rms.header.remove(key, ignore_missing=True)
        hdulist_combined = fits.HDUList([hdu_pri, hdu_sci, hdu_rms])
        os.makedirs(self.out_path, exist_ok=True)
        hdulist_combined.writeto(self.out_path / out_fn, overwrite=self.overwrite)

## Example

The following is an example of combining a set of dithers.

In [ ]:
path = os.path.expanduser("~ppzjbg/Q1_ICL/Q1_data/allQ1data")
obs_id = 2682

In [ ]:
%%time
combiner = Combiner(in_path=path, out_path=os.path.join(path, "../stacked"), debug=True)
combiner.combine(obs_ids=obs_id, filters="Y")

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()